In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

# Use standard ROOT/CMS style
hep.style.use(hep.style.ROOT)

def draw_epic_labels(ax):
    """Draws the standard ePIC performance block in the top left corner."""
    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes, 
            fontsize=12, fontweight='bold', ha='left', va='top')
    ax.text(0.04, 0.88, "26.03.1 Campaign", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top')
    ax.text(0.04, 0.82, r"e+p $\Upsilon_{\gamma p}$ 18x275 GeV$^2$", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top')
    ax.text(0.04, 0.76, r"$\mathcal{L}_{proj} = 10$ fb$^{-1}$, $\sqrt{s} = 141$ GeV", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top')

def main():
    filename = "~/merged_elecs_test_tuples.root" 
    expected_lumi_invfb = 10.0  # Must match the lumi used in the C++ script!
    
    try:
        df = ROOT.RDataFrame("UpsilonTree", filename)
    except Exception as e:
        print(f"Error opening file: {e}")
        return

    # 1. Extract the Truth-Level Data (No detector cuts, just what was generated)
    data_true = df.Filter("true_mass > 0").AsNumpy(["true_y", "weight"])
    
    # 2. Extract the Reco-Level Data (Events that survived all cuts)
    df_reco = df.Filter("reco_mass >= 8.0 && reco_mass <= 12.0") \
                .Filter("abs(sigma_epz - 36.0) <= 3.6") \
                .Filter("reco_W >= 20.0")
    data_reco = df_reco.AsNumpy(["reco_y", "weight"])

    # Define bins
    y_bins = np.linspace(-4, 4, 32)
    bin_width = y_bins[1] - y_bins[0]
    bin_centers = (y_bins[:-1] + y_bins[1:]) / 2.0

    # =========================================================================
    # CALCULATE CROSS SECTIONS
    # Expected Yield in Bin = Sum of Weights
    # dSigma/dy = Yield / (Luminosity * bin_width)
    # =========================================================================
    
    # Truth Cross Section
    counts_true, _ = np.histogram(data_true["true_y"], bins=y_bins, weights=data_true["weight"])
    vars_true, _   = np.histogram(data_true["true_y"], bins=y_bins, weights=data_true["weight"]**2)
    
    dsigma_dy_true = counts_true / (expected_lumi_invfb * bin_width)
    err_dsigma_dy_true = np.sqrt(vars_true) / (expected_lumi_invfb * bin_width)

    # Reconstructed Visible Cross Section (Uncorrected)
    counts_reco, _ = np.histogram(data_reco["reco_y"], bins=y_bins, weights=data_reco["weight"])
    vars_reco, _   = np.histogram(data_reco["reco_y"], bins=y_bins, weights=data_reco["weight"]**2)
    
    dsigma_dy_reco = counts_reco / (expected_lumi_invfb * bin_width)
    err_dsigma_dy_reco = np.sqrt(vars_reco) / (expected_lumi_invfb * bin_width)

    # =========================================================================
    # PLOT
    # =========================================================================
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Draw Truth as a solid black step line
    hep.histplot(dsigma_dy_true, bins=y_bins, histtype='step', 
                 color='black', label=r'Generated $\frac{d\sigma}{dy}$ (Truth)', linewidth=2)
                 
    # Draw Reco as blue data points
    ax.errorbar(bin_centers, dsigma_dy_reco, yerr=err_dsigma_dy_reco, fmt='o', 
                color='blue', label=r'Visible $\frac{d\sigma}{dy}$ (Reco, uncorrected)', markersize=6)

    ax.set_xlabel(r"Rapidity $y$")
    ax.set_ylabel(r"$\frac{d\sigma}{dy}$ [fb]")
    
    draw_epic_labels(ax)
    
    # Expand y-axis limits to accommodate the text block
    if np.max(dsigma_dy_true) > 0:
        ax.set_ylim(0, np.max(dsigma_dy_true) * 1.5)
        
    ax.legend(loc='upper right', fontsize=12)
        
    fig.tight_layout()
    fig.savefig("Upsilon_dsigma_dy.png", dpi=300)
    print("Saved Upsilon_dsigma_dy.png")

if __name__ == "__main__":
    main()